# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the record sets defined in the dataset. Each record set, field, and column is referenced by its `@id`. This ensures clear identification for downstream analysis.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print('Available Record Sets (@id):')
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '[Unnamed]')}")

# For each record set, list its fields (@id)
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"  Field @id: {field['@id']} | Name: {field.get('name', '[Unnamed]')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all records from the main record set. Please update the record set `@id`s as required if more than one is available.

In [ ]:
# Extract data from each record set
# We'll use the first record set found for illustration; if there are more, you can add more ids to the list.
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns and first few records from the primary record set
primary_rs_id = record_set_ids[0]
print(f"Fields for Record Set {primary_rs_id}:")
print(dataframes[primary_rs_id].columns.tolist())
dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (e.g., GAD-7 score) referenced by its `@id`, and perform filtering, normalization, and grouping. Please refer to the field list above for actual `@id` values.

In [ ]:
# EDA Example: Let's use the GAD-7 score field (replace with actual @id if different)
# For demonstration, we look for a field whose name contains 'GAD' or 'Score'
primary_fields = record_sets[0].get('fields', [])
numeric_field_id = None
for field in primary_fields:
    if 'gad' in field.get('name', '').lower() or 'score' in field.get('name', '').lower():
        numeric_field_id = field['@id']
        break

if numeric_field_id is None:
    # If not found, pick the first numeric field by type
    for field in primary_fields:
        if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
            numeric_field_id = field['@id']
            break

print(f"Using numeric field @id: {numeric_field_id}")

df = dataframes[primary_rs_id]

# Filtering: Keep records where value > threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping: Find a categorical/group field (e.g., gender or education)
    group_field_id = None
    for field in primary_fields:
        if field.get('dataType', '').lower() == 'text' and (
            'gender' in field.get('name', '').lower() or 'education' in field.get('name', '').lower()
        ):
            group_field_id = field['@id']
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean values of {numeric_field_id} by {group_field_id}:\n", grouped_df.head())
else:
    print("No suitable numeric field found in record set columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the normalized numeric field (e.g., GAD-7 score), and, if available, also visualize group-wise averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution - histogram
if numeric_field_id and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f'Normalized Distribution of {numeric_field_id}')
    plt.xlabel('Normalized Value')
    plt.ylabel('Frequency')
    plt.show()

# Plot groupwise means if group_field_id and grouped_df exist
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 5))
    sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded using the `mlcroissant` library and explored via its Croissant schema.
- Record sets, fields, and columns were referenced by their `@id`, enabling consistent, reproducible analysis.
- Numeric fields such as GAD-7 scores were filtered, normalized, and visualized. Demographic groupings were shown if available.
- This approach demonstrates the flexibility of working with standardized metadata and record structures for AI-ready datasets.